Enrollment No.:240840116005

Practical-10 Mini Project: Analyze a real-world dataset, build an interactive dashboard, and present actionable insights with ethical
considerations.


## Cell 1 — Install required libraries


In [1]:
!pip -q install dash plotly pandas numpy

## Cell 2 — Import libraries

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from google.colab import files
from dash import Dash, dcc, html, Input, Output
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


## Cell 3 — Upload your dataset (optional)



In [3]:
# Set this to True if you want to upload a CSV file.
UPLOAD_REAL_CSV = False

uploaded_df = None
if UPLOAD_REAL_CSV:
    uploaded = files.upload()
    if uploaded:
        file_name = next(iter(uploaded))
        uploaded_df = pd.read_csv(file_name)
        print("Uploaded:", file_name)
        display(uploaded_df.head())
else:
    print("No CSV uploaded. The notebook will use simulated practice data.")

No CSV uploaded. The notebook will use simulated practice data.


## Cell 4 — Create a practice dataset if no CSV was uploaded



In [4]:
if uploaded_df is None:
    np.random.seed(42)
    n = 500
    event_types = ["Concert", "Workshop", "Seminar", "Sports", "Exhibition"]
    locations = ["Ahmedabad", "Surat", "Vadodara", "Mumbai", "Pune", "Delhi"]
    event_names = [
        "Music Night", "Python Workshop", "Business Seminar",
        "Football Tournament", "Art Exhibition", "Tech Conference",
        "Startup Meetup", "Dance Festival", "Career Fair", "Design Workshop"
    ]

    dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")
    df = pd.DataFrame({
        "Event_ID": [f"EV{i:04d}" for i in range(1, n + 1)],
        "Event_Name": np.random.choice(event_names, n),
        "Event_Type": np.random.choice(event_types, n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "Location": np.random.choice(locations, n),
        "Event_Date": np.random.choice(dates, n),
        "Total_Seats": np.random.randint(100, 1001, n),
        "Ticket_Price": np.random.choice([100, 150, 200, 250, 300, 500, 750, 1000], n)
    })
    df["Tickets_Sold"] = [
        np.random.randint(20, seats + 1) for seats in df["Total_Seats"]
    ]
    df["Attendance"] = [
        np.random.randint(int(sold * 0.65), sold + 1) for sold in df["Tickets_Sold"]
    ]
    df["Revenue"] = df["Tickets_Sold"] * df["Ticket_Price"]
    df["Rating"] = np.round(np.random.uniform(2.5, 5.0, n), 1)
    DATA_SOURCE = "Simulated practice data — replace with a documented public dataset"
else:
    df = uploaded_df.copy()
    DATA_SOURCE = "User-uploaded CSV — verify and document its original source"

print("Data source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

Data source: Simulated practice data — replace with a documented public dataset
Dataset shape: (500, 11)


,Event_ID,Event_Name,Event_Type,Location,Event_Date,Total_Seats,Ticket_Price,Tickets_Sold,Attendance,Revenue,Rating
0,EV0001,Startup Meetup,Concert,Pune,2024-01-21,696,200,543,520,108600,3.5
1,EV0002,Football Tournament,Exhibition,Vadodara,2024-10-29,757,150,35,35,5250,4.8
2,EV0003,Dance Festival,Seminar,Pune,2024-03-24,934,750,148,107,111000,5.0
3,EV0004,Art Exhibition,Sports,Vadodara,2025-10-20,653,750,594,455,445500,2.6
4,EV0005,Startup Meetup,Workshop,Mumbai,2025-03-28,473,200,128,99,25600,4.4


## Cell 5 — Standardize column names and check required fields



In [5]:
# Edit the left-side names to match the column headers in your CSV.
rename_map = {
    # "event_category": "Event_Type",
    # "date": "Event_Date",
    # "tickets": "Tickets_Sold",
    # "attendees": "Attendance",
    # "sales": "Revenue",
}
df = df.rename(columns=rename_map)

required = ["Event_Type", "Event_Date", "Tickets_Sold", "Attendance"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Please rename your CSV columns in rename_map (Cell 5)."
    )

if "Event_Name" not in df.columns:
    df["Event_Name"] = "Event"
if "Location" not in df.columns:
    df["Location"] = "Not specified"
if "Ticket_Price" not in df.columns:
    df["Ticket_Price"] = np.nan
if "Revenue" not in df.columns:
    df["Revenue"] = df["Tickets_Sold"] * df["Ticket_Price"]

print("Column check completed.")
print(df.columns.tolist())

Column check completed.
['Event_ID', 'Event_Name', 'Event_Type', 'Location', 'Event_Date', 'Total_Seats', 'Ticket_Price', 'Tickets_Sold', 'Attendance', 'Revenue', 'Rating']


## Cell 6 — Data cleaning
.

In [6]:
df["Event_Date"] = pd.to_datetime(df["Event_Date"], errors="coerce")
numeric_cols = ["Tickets_Sold", "Attendance", "Revenue", "Ticket_Price", "Total_Seats", "Rating"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["Event_Type", "Event_Date", "Tickets_Sold", "Attendance"])
df["Event_Type"] = df["Event_Type"].astype(str).str.strip().str.title()
df["Tickets_Sold"] = df["Tickets_Sold"].clip(lower=0)
df["Attendance"] = df["Attendance"].clip(lower=0)
df["Month"] = df["Event_Date"].dt.to_period("M").astype(str)

print("Duplicate rows removed:", before - len(df))
print("Remaining rows:", len(df))
print("Missing values after essential cleaning:")
display(df.isna().sum().to_frame("Missing values"))

Duplicate rows removed: 0
Remaining rows: 500
Missing values after essential cleaning:


,Missing values
Event_ID,0
Event_Name,0
Event_Type,0
Location,0
Event_Date,0
Total_Seats,0
Ticket_Price,0
Tickets_Sold,0
Attendance,0
Revenue,0


## Cell 7 — Create useful analysis measures

In [7]:
# Attendance rate = people who attended / tickets sold.
df["Attendance_Rate"] = np.where(
    df["Tickets_Sold"] > 0,
    df["Attendance"] / df["Tickets_Sold"] * 100,
    0
)

# Ticket sell-through rate is available when seat capacity is provided.
if "Total_Seats" in df.columns:
    df["Occupancy_Rate"] = np.where(
        df["Total_Seats"] > 0,
        df["Tickets_Sold"] / df["Total_Seats"] * 100,
        np.nan
    )

print("Calculated attendance and occupancy measures.")
display(df[["Event_Name", "Event_Type", "Tickets_Sold", "Attendance", "Attendance_Rate"]].head())

Calculated attendance and occupancy measures.


,Event_Name,Event_Type,Tickets_Sold,Attendance,Attendance_Rate
0,Startup Meetup,Concert,543,520,95.764273
1,Football Tournament,Exhibition,35,35,100.000000
2,Dance Festival,Seminar,148,107,72.297297
3,Art Exhibition,Sports,594,455,76.599327
4,Startup Meetup,Workshop,128,99,77.343750


## Cell 8 — Exploratory Data Analysis (EDA)

In [8]:
print("Dataset summary")
display(df.describe(include="all").T)

print("\nEvents by category")
display(df["Event_Type"].value_counts().rename_axis("Event Type").to_frame("Number of events"))

print("\nTop event locations")
display(df["Location"].value_counts().head(10).rename_axis("Location").to_frame("Events"))

Dataset summary


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Event_ID,500,500,EV0500,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event_Name,500,10,Music Night,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event_Type,500,5,Concert,139,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,500,6,Pune,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event_Date,500,NaN,NaN,NaN,2024-12-28 16:56:38.400000256,2024-01-01 00:00:00,2024-06-18 18:00:00,2024-12-16 12:00:00,2025-07-06 00:00:00,2025-12-29 00:00:00,NaN
Total_Seats,500.0,NaN,NaN,NaN,557.612,106.0,337.75,563.0,761.0,1000.0,246.484284
Ticket_Price,500.0,NaN,NaN,NaN,412.4,100.0,200.0,250.0,750.0,1000.0,303.133667
Tickets_Sold,500.0,NaN,NaN,NaN,304.378,20.0,128.0,265.5,443.5,918.0,210.602521
Attendance,500.0,NaN,NaN,NaN,250.196,13.0,106.5,212.0,363.0,853.0,176.345073
Revenue,500.0,NaN,NaN,NaN,122451.1,2000.0,33075.0,75550.0,161062.5,817000.0,127737.159894



Events by category


,Number of events
Event Type,
Concert,139
Workshop,117
Seminar,100
Exhibition,80
Sports,64



Top event locations


,Events
Location,
Pune,93
Surat,92
Vadodara,89
Ahmedabad,82
Delhi,73
Mumbai,71


## Cell 9 — Prepare summary tables

In [9]:
category_summary = (
    df.groupby("Event_Type", as_index=False)
      .agg(Events=("Event_ID", "count") if "Event_ID" in df.columns else ("Event_Name", "count"),
           Tickets_Sold=("Tickets_Sold", "sum"),
           Attendance=("Attendance", "sum"),
           Revenue=("Revenue", "sum"))
      .sort_values("Events", ascending=False)
)
monthly_summary = (
    df.groupby("Month", as_index=False)
      .agg(Tickets_Sold=("Tickets_Sold", "sum"), Revenue=("Revenue", "sum"),
           Attendance=("Attendance", "sum"))
      .sort_values("Month")
)
display(category_summary)
display(monthly_summary.head())

,Event_Type,Events,Tickets_Sold,Attendance,Revenue
0,Concert,139,43994,36977,17135950
4,Workshop,117,36031,29364,15398900
2,Seminar,100,25582,20524,10643900
1,Exhibition,80,24073,19382,8413200
3,Sports,64,22509,18851,9633600


,Month,Tickets_Sold,Revenue,Attendance
0,2024-01,9309,3454750,7644
1,2024-02,4392,2003750,3692
2,2024-03,6473,3232400,5277
3,2024-04,6054,2286050,4932
4,2024-05,7058,3197750,5785


## Cell 10 — Static charts for initial analysis

In [15]:
fig1 = px.pie(category_summary, names="Event_Type", values="Events",
               title="Event Distribution by Category", hole=0.35)
fig1.show()

fig2 = px.bar(monthly_summary, x="Month", y="Tickets_Sold",
              title="Monthly Ticket Sales", labels={"Month": "Month"})
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

fig3 = px.line(monthly_summary, x="Month", y="Revenue",
               markers=True, title="Monthly Ticket Revenue")
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

attendance_summary = df.groupby("Event_Type", as_index=False).agg(
    Registered=("Tickets_Sold", "sum"), Attended=("Attendance", "sum")
)
fig4 = px.bar(attendance_summary, x="Event_Type", y=["Registered", "Attended"],
              barmode="group", title="Registered vs Attended by Event Type")
fig4.show()

## Cell 11 — Build the interactive Dash dashboard

Use the dropdown to filter event category. The four charts update based on the selected category. The `jupyter_mode="external"` option provides a link to open the dashboard from Colab.

In [11]:
app = Dash(__name__)
event_options = [{"label": "All Event Types", "value": "All"}] + [
    {"label": x, "value": x} for x in sorted(df["Event_Type"].dropna().unique())
]

app.layout = html.Div([
    html.H1("Event Management Analytics Dashboard",
            style={"textAlign": "center", "color": "#17324D"}),
    html.P("Explore ticket sales, revenue and event attendance.",
           style={"textAlign": "center"}),
    html.Div([
        html.Label("Select Event Category:"),
        dcc.Dropdown(id="event-filter", options=event_options,
                     value="All", clearable=False,
                     style={"color": "#222", "maxWidth": "500px"})
    ], style={"padding": "15px"}),
    html.Div(id="kpi-cards",
             style={"display": "flex", "gap": "15px", "flexWrap": "wrap",
                    "justifyContent": "space-around"}),
    html.Div([
        dcc.Graph(id="category-chart"),
        dcc.Graph(id="sales-chart"),
        dcc.Graph(id="revenue-chart"),
        dcc.Graph(id="attendance-chart")
    ], style={"display": "grid",
              "gridTemplateColumns": "repeat(auto-fit, minmax(350px, 1fr))",
              "gap": "12px", "padding": "10px"})
], style={"fontFamily": "Arial", "backgroundColor": "#F4F7FA",
          "padding": "15px"})

@app.callback(
    Output("kpi-cards", "children"),
    Output("category-chart", "figure"),
    Output("sales-chart", "figure"),
    Output("revenue-chart", "figure"),
    Output("attendance-chart", "figure"),
    Input("event-filter", "value")
)
def update_dashboard(selected_type):
    filtered = df if selected_type == "All" else df[df["Event_Type"] == selected_type]

    total_events = len(filtered)
    total_tickets = int(filtered["Tickets_Sold"].sum())
    total_attendance = int(filtered["Attendance"].sum())
    total_revenue = filtered["Revenue"].sum(min_count=1)
    revenue_text = "N/A" if pd.isna(total_revenue) else f"₹{total_revenue:,.0f}"

    def kpi(title, value):
        return html.Div([
            html.H4(title, style={"marginBottom": "5px"}),
            html.H2(value, style={"marginTop": "0", "color": "#176B87"})
        ], style={"background": "white", "padding": "15px",
                  "borderRadius": "10px", "minWidth": "180px",
                  "boxShadow": "0 1px 5px #D5DDE5"})

    cards = [
        kpi("Total Events", f"{total_events:,}"),
        kpi("Tickets Sold", f"{total_tickets:,}"),
        kpi("Attendees", f"{total_attendance:,}"),
        kpi("Ticket Revenue", revenue_text)
    ]

    cat = filtered.groupby("Event_Type", as_index=False).size().rename(columns={"size": "Events"})
    f1 = px.pie(cat, names="Event_Type", values="Events",
                title="Event Category Distribution", hole=0.35)

    month = filtered.assign(Month=filtered["Event_Date"].dt.to_period("M").astype(str))
    month = month.groupby("Month", as_index=False).agg(
        Tickets_Sold=("Tickets_Sold", "sum"), Revenue=("Revenue", "sum")
    ).sort_values("Month")
    f2 = px.bar(month, x="Month", y="Tickets_Sold", title="Monthly Ticket Sales")
    f2.update_layout(xaxis_tickangle=-45)

    f3 = px.line(month, x="Month", y="Revenue", markers=True,
                 title="Monthly Ticket Revenue")
    f3.update_layout(xaxis_tickangle=-45, yaxis_title="Revenue (₹)")

    att = filtered.groupby("Event_Type", as_index=False).agg(
        Registered=("Tickets_Sold", "sum"), Attended=("Attendance", "sum")
    )
    f4 = px.bar(att, x="Event_Type", y=["Registered", "Attended"],
                barmode="group", title="Registered vs Attended Participants")

    for fig in [f1, f2, f3, f4]:
        fig.update_layout(template="plotly_white", margin=dict(l=30, r=20, t=60, b=40))

    return cards, f1, f2, f3, f4

print("Dashboard app is ready.")

Dashboard app is ready.


## Cell 12 — Run the dashboard
Run this cell and open the external Dash link shown by Colab.

In [12]:
app.run(jupyter_mode="external", debug=False, port=8050)

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Cell 13 — Generate actionable insights

These observations are calculated from the currently loaded dataset. Interpret them in context and do not claim that correlation proves causation.

In [13]:
if len(df):
    best_category = df.groupby("Event_Type")["Tickets_Sold"].sum().idxmax()
    best_month = (df.assign(Month=df["Event_Date"].dt.to_period("M").astype(str))
                    .groupby("Month")["Tickets_Sold"].sum().idxmax())
    best_location = df.groupby("Location")["Tickets_Sold"].sum().idxmax()
    attendance_rate = (df["Attendance"].sum() / df["Tickets_Sold"].sum() * 100
                       if df["Tickets_Sold"].sum() else 0)

    print("ACTIONABLE INSIGHTS")
    print(f"1. Highest ticket-sales category: {best_category}")
    print(f"2. Month with highest ticket sales: {best_month}")
    print(f"3. Location with highest ticket sales: {best_location}")
    print(f"4. Overall attendance-to-ticket-sales rate: {attendance_rate:.1f}%")
    print("\nSuggested actions:")
    print("- Review high-performing categories when planning future events.")
    print("- Investigate low attendance events and improve reminders or event communication.")
    print("- Compare locations only after considering event size, price and promotion.")
    print("- Use monthly trends to plan marketing campaigns.")
else:
    print("No records available for insights.")

ACTIONABLE INSIGHTS
1. Highest ticket-sales category: Concert
2. Month with highest ticket sales: 2024-01
3. Location with highest ticket sales: Surat
4. Overall attendance-to-ticket-sales rate: 82.2%

Suggested actions:
- Review high-performing categories when planning future events.
- Investigate low attendance events and improve reminders or event communication.
- Compare locations only after considering event size, price and promotion.
- Use monthly trends to plan marketing campaigns.


## Cell 14 — Ethical considerations and limitations

Include these points in your report and presentation:

1. **Privacy:** Remove names, email addresses, phone numbers and other personal identifiers before analysis.
2. **Data quality:** Document missing values, duplicates, inconsistent labels and assumptions.
3. **Bias and representation:** A dataset from a limited region or platform may not represent all event audiences.
4. **Responsible interpretation:** Ticket sales are not the same as actual attendance, satisfaction or event success.
5. **Transparency:** State whether the dataset is public, collected with permission, or simulated.
6. **Security:** Do not publish raw attendee-level personal data in a public dashboard.

**Limitations:** If the dataset does not include marketing spend, refunds, costs or customer feedback, this analysis cannot calculate profit, campaign return on investment or explain why people chose not to attend.

## Cell 15 — Export the cleaned dataset and summary tables

In [14]:
df.to_csv("cleaned_event_management_data.csv", index=False)
category_summary.to_csv("event_category_summary.csv", index=False)
monthly_summary.to_csv("monthly_event_summary.csv", index=False)
print("Exported cleaned data and summary CSV files.")
files.download("cleaned_event_management_data.csv")

Exported cleaned data and summary CSV files.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This mini-project demonstrates how event data can be cleaned, analyzed and communicated through an interactive dashboard. The dashboard helps event organizers monitor ticket sales, revenue and attendance, compare event categories, and identify areas for improvement.

**Before final submission:** Replace simulated data with a documented real-world dataset if the assignment specifically requires real-world data. Record the dataset source, access date, license and any preprocessing steps.